In [1]:
!pip install -q ultralytics opencv-python
import os, glob, random, gc
from collections import defaultdict
import yaml
import torch
import pandas as pd
from PIL import Image
from ultralytics import YOLO
print("Setup Complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Setup Complete.


In [2]:
DATASET_PATH = "/kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data"
IMAGES_DIR = os.path.join(DATASET_PATH, "train/images")
LABELS_DIR = os.path.join(DATASET_PATH, "train/labels")

class_names = {
    0: "aegypti",
    1: "albopictus",
    2: "anopheles",
    3: "culex",
    4: "culiseta",
    5: "japonicus/koreicus"
}

In [3]:
random.seed(42)

label_files = glob.glob(os.path.join(LABELS_DIR, "*.txt"))
class_to_images = defaultdict(list)
img_to_classes = {}

for lf in label_files:
    base = os.path.splitext(os.path.basename(lf))[0]
    with open(lf) as f:
        classes_in_img = {int(line.split()[0]) for line in f if line.strip()}
    img_to_classes[base] = classes_in_img
    for c in classes_in_img:
        class_to_images[c].append(base)

val_frac = 0.20
val_set = set()
for c, imgs in class_to_images.items():
    imgs = list(set(imgs))
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * val_frac))  # at least 1 val image even for rare classes
    val_set.update(imgs[:n_val])

all_basenames = list(img_to_classes.keys())
train_set = [b for b in all_basenames if b not in val_set]

print(f"total images: {len(all_basenames)} | train: {len(train_set)} | val: {len(val_set)}")

val_class_counts = defaultdict(int)
for b in val_set:
    for c in img_to_classes[b]:
        val_class_counts[c] += 1
for c in sorted(class_names):
    print(f"  val instances for class {c} ({class_names[c]}): {val_class_counts[c]}")

total images: 7500 | train: 6002 | val: 1498
  val instances for class 0 (aegypti): 7
  val instances for class 1 (albopictus): 666
  val instances for class 2 (anopheles): 11
  val instances for class 3 (culex): 662
  val instances for class 4 (culiseta): 92
  val instances for class 5 (japonicus/koreicus): 60


In [4]:
def find_ext(base):
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(IMAGES_DIR, base + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(base)

os.makedirs('/kaggle/working/data', exist_ok=True)
train_txt = "/kaggle/working/data/train.txt"
val_txt = "/kaggle/working/data/val.txt"

with open(train_txt, "w") as f:
    f.write("\n".join(find_ext(b) for b in train_set))
with open(val_txt, "w") as f:
    f.write("\n".join(find_ext(b) for b in val_set))

print("train.txt / val.txt written")

train.txt / val.txt written


In [5]:
yaml_content = {
    'path': DATASET_PATH,
    'train': train_txt,
    'val': val_txt,
    'names': class_names
}
yaml_path = '/kaggle/working/data/mosquito.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)
print(f"Dataset YAML created at: {yaml_path}")

Dataset YAML created at: /kaggle/working/data/mosquito.yaml


In [6]:
model = YOLO('yolo11m.pt')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=32,
    workers=4,
    #cache=True,
    amp=True,
    val=True,
    #patience=5,
    device=0,
    lr0=0.001,
    lrf=0.2,
    momentum=0.937,
    weight_decay=0.0005,
    pretrained=True,
    hsv_h= 0.015,
    hsv_s= 0.7,
    hsv_v= 0.4,
    degrees= 0.5,
    translate= 0.1,
    scale= 0.5,
    shear= 0.1,
    mosaic= 1.0,
    mixup= 0.2,
    project='/kaggle/working/runs',
    name='mosquito_yolo',
    exist_ok=True
)

print("Training finished successfully.")

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data/mosquito.yaml, degrees=0.5, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.2, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mosquito_yolo, nbs=64, nms=

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


train: Scanning /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train/labels... 6002 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6002/6002 302.8it/s 19.8s
WARNING ⚠️ train: Cache directory /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train is not writable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.3 ms, read: 35.8±34.3 MB/s, size: 1249.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train/labels... 1498 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1498/1498 265.0it/s 

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


train: Scanning /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train/labels... 6002 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6002/6002 1.1Kit/s 5.4s
WARNING ⚠️ train: Cache directory /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train is not writable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 392.5±266.3 MB/s, size: 483.1 KB)
val: Scanning /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train/labels... 1498 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1498/1498 978.2it/s 1.5s
WARNING ⚠️ val: Cache directory /kaggle/input/competitions/dlp-26t2-week10-assignment/final_dlp_data/final_dlp_data/train is not writable, cache not sa

In [7]:
if hasattr(model, "trainer"):
    del model.trainer
gc.collect()
torch.cuda.empty_cache()

WEIGHTS_PATH = "/kaggle/working/runs/mosquito_yolo/weights/best.pt"
model = YOLO(WEIGHTS_PATH)

Exception in thread Thread-19 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

In [8]:
test_img_dir = os.path.join(DATASET_PATH, "test/images")
test_image_paths = sorted(
    glob.glob(os.path.join(test_img_dir, "*.jpg")) +
    glob.glob(os.path.join(test_img_dir, "*.png")) +
    glob.glob(os.path.join(test_img_dir, "*.jpeg"))
)
print("test images:", len(test_image_paths))

majority_fallback = "albopictus"
submission_rows = []

BATCH = 25
for i in range(0, len(test_image_paths), BATCH):
    chunk = test_image_paths[i:i + BATCH]
    chunk_results = model.predict(
        source=chunk,
        conf=0.3,
        iou=0.5,
        imgsz=640,
        device=0,
        verbose=False
    )
    for res in chunk_results:
        image_id = os.path.basename(res.path)
        boxes = res.boxes

        if boxes is not None and len(boxes) > 0:
            confs = boxes.conf.cpu().numpy()
            best_idx = confs.argmax()
            xc, yc, w, h = boxes.xywhn[best_idx].cpu().tolist()
            cls_id = int(boxes.cls[best_idx].cpu().item())
            submission_rows.append({
                "ImageID": image_id,
                "LabelName": class_names[cls_id],
                "Conf": float(confs[best_idx]),
                "xcenter": xc, "ycenter": yc,
                "bbx_width": w, "bbx_height": h
            })
        else:
            submission_rows.append({
                "ImageID": image_id,
                "LabelName": majority_fallback,
                "Conf": 0.01,
                "xcenter": 0.5, "ycenter": 0.5,
                "bbx_width": 0.1, "bbx_height": 0.1
            })

    del chunk_results
    gc.collect()
    torch.cuda.empty_cache()
    print(f"processed {min(i+BATCH, len(test_image_paths))}/{len(test_image_paths)}")

test images: 525
processed 25/525
processed 50/525
processed 75/525
processed 100/525
processed 125/525
processed 150/525
processed 175/525
processed 200/525
processed 225/525
processed 250/525
processed 275/525
processed 300/525
processed 325/525
processed 350/525
processed 375/525
processed 400/525
processed 425/525
processed 450/525
processed 475/525
processed 500/525
processed 525/525


In [9]:
sub_df = pd.DataFrame(submission_rows)
sub_df = sub_df.loc[sub_df.groupby("ImageID")["Conf"].idxmax()].reset_index(drop=True)
sub_df.insert(0, "id", range(len(sub_df)))
sub_df = sub_df[["id", "ImageID", "LabelName", "Conf", "xcenter", "ycenter", "bbx_width", "bbx_height"]]

output_path = "/kaggle/working/submission.csv"
sub_df.to_csv(output_path, index=False)

assert len(sub_df) == len(test_image_paths), f"row mismatch: {len(sub_df)} vs {len(test_image_paths)}"
print(f"Submission generated successfully with exactly {len(sub_df)} rows at {output_path}.")
sub_df.head(10)

Submission generated successfully with exactly 525 rows at /kaggle/working/submission.csv.


,id,ImageID,LabelName,Conf,xcenter,ycenter,bbx_width,bbx_height
0,0,0031063e-716a-4080-934c-77598dc8de72.jpeg,culex,0.863320,0.496562,0.560224,0.151770,0.122089
1,1,00fbfad7-9722-4581-831c-79faa576ea7f.jpeg,japonicus/koreicus,0.362662,0.476699,0.616827,0.193957,0.159445
2,2,02043b0e-3d7d-4ca4-a36f-4bf97c344264.jpeg,culex,0.846441,0.596599,0.466212,0.504093,0.430875
3,3,0365513c-8f00-44f3-abd0-1fadde81c602.jpeg,culex,0.900893,0.440783,0.746347,0.402831,0.199360
4,4,0365d78b-2064-4d54-b15c-994d1950479a.jpeg,albopictus,0.709148,0.233216,0.203600,0.119901,0.087389
5,5,03d6749a-c5b3-45a3-81fa-b9c2f09eb4ba.jpeg,albopictus,0.738882,0.501020,0.524535,0.628506,0.767913
6,6,03ef458e-8f40-4d42-811b-f3c2db667e12.jpeg,albopictus,0.696271,0.270393,0.647358,0.316297,0.440574
7,7,04741ad4-6e1c-416d-acd4-b352cf18ebda.jpeg,culex,0.810104,0.579700,0.420464,0.146193,0.119619
8,8,04a95170-0348-43e1-a722-e178370e6812.jpeg,albopictus,0.331131,0.077477,0.318890,0.146121,0.080902
9,9,04d4d98b-4b4d-45d1-924a-82f4c3a9ff27.jpeg,culex,0.868121,0.457094,0.388011,0.166486,0.171622
